## Step 0: Import libraries

In [ ]:
import os
import shutil
import tempfile
from pyspark.sql import SparkSession, Row

## Step 1: Start Spark and create the DataFrame

In [ ]:
spark = (SparkSession.builder.appName("DS505-Practical08").master("local[*]")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

data = [
    {"Name": "Aarav", "Department": "IT",      "City": "Surat",  "Salary": 72000, "Rating": 4.5},
    {"Name": "Diya",  "Department": "HR",      "City": "Mumbai", "Salary": 54000, "Rating": 4.1},
    {"Name": "Kabir", "Department": "Finance", "City": "Surat",  "Salary": 65000, "Rating": 3.9},
    {"Name": "Meera", "Department": "IT",      "City": "Pune",   "Salary": 88000, "Rating": 4.8},
    {"Name": "Rohan", "Department": "Sales",   "City": "Surat",  "Salary": 48000, "Rating": 3.6},
]
df = spark.createDataFrame([Row(**d) for d in data])
df.printSchema()
df.show()

## Step 2: Export to Parquet – `.write.mode('overwrite').parquet()`

In [ ]:
tmp_dir = tempfile.mkdtemp()
path = os.path.join(tmp_dir, "employees.parquet")

df.write.mode("overwrite").parquet(path)
print("Parquet written to:", path)
print("Files inside      :", [f for f in os.listdir(path) if not f.startswith(".")])

## Step 3: Reload – `spark.read.parquet()`

In [ ]:
df_back = spark.read.parquet(path)
df_back.printSchema()
df_back.show()

## Step 4: Verify schema preservation and row counts

In [ ]:
print("Schema preserved :", df.schema == df_back.schema)
print(f"Row count original = {df.count()} | reloaded = {df_back.count()} | equal = {df.count() == df_back.count()}")

## Step 5 (bonus): Column pruning – the advantage of columnar storage
Parquet lets Spark read only the columns it needs.

In [ ]:
df_back.select("Salary").show()

## Step 6: Clean up and stop the session

In [ ]:
shutil.rmtree(tmp_dir)
spark.stop()
print("Cleaned up.")